# Final evaluation and error analysis — Step 13

**CSE437 Data Science | Group 15**

This notebook inspects the completed official held-out evaluation. The selected-feature Logistic Regression pipeline was frozen in Step 12 (`C=1`, `class_weight=balanced`, threshold 0.5), fitted on all 95,415 development rows, and evaluated once on 23,795 later-arrival test rows. The original dataset, target, problem/questions and split are unchanged.

The official evaluation is implemented in `src/final_evaluation.py`; its protocol was saved before test-label access. The cells below load and verify the resulting artifacts rather than select or change a model. This notebook does not regenerate final-test evidence or promote notebook 04's rerun selection. Missing saved comparison artifacts cause an error rather than triggering model training.

**Execution provenance:** all six code cells below executed sequentially in a fresh Python process with actual stdout captured. A fresh Jupyter-kernel run and canonical nbformat validation remain final submission gates.


## Run this notebook

[Open in Google Colab](https://colab.research.google.com/github/faraaz1027-cloud/cse437-hotel-cancellation-15/blob/main/notebooks/05_evaluation_and_error_analysis.ipynb)

In Colab, choose **Runtime > Run all**. The setup cell downloads the full project, checks the original inputs and installs pinned analysis packages before the analysis starts. No manual upload or Drive mount is required; a CPU runtime is sufficient. If setup says **SETUP PAUSED**, choose **Runtime > Restart session**, then **Runtime > Run all** again. Do not delete the runtime; the installed packages remain.

The automatic download uses analysis snapshot `e01e785b78f2849b423c5be4a3fe5221a96f3e66`. These setup cells were added later; existing analysis cells and recorded outputs are unchanged. A local Jupyter run keeps its existing environment and repository. Each notebook can use the committed inputs independently; read notebooks 01–05 in order.

Notebook 04 can take several minutes and may report numerical reproduction differences. Notebook 05 verifies cached final-test evidence, not a new model fit. Download your executed notebook if you want to keep new outputs. Colab cloud execution of this setup has not yet been independently verified.

In [ ]:
"""Self-contained Colab bootstrap embedded verbatim in the five notebooks.

Uses only the standard library until the analysis dependencies are ready.
Does not modify a user's checkout, install Jupyter, or weaken scientific checks.
"""
import hashlib
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys


SOURCE_URL = 'https://github.com/faraaz1027-cloud/cse437-hotel-cancellation-15.git'
SOURCE_COMMIT = 'e01e785b78f2849b423c5be4a3fe5221a96f3e66'
ANALYSIS_PACKAGES = {
    'numpy': ('numpy', '2.3.5'),
    'pandas': ('pandas', '2.2.3'),
    'scipy': ('scipy', '1.17.0'),
    'scikit-learn': ('sklearn', '1.8.0'),
    'matplotlib': ('matplotlib', '3.10.8'),
    'seaborn': ('seaborn', '0.13.2'),
    'joblib': ('joblib', '1.5.3'),
}
PROTECTED_FILES = {
    'data/raw/hotel_bookings.csv':
        '7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06',
    'data/results/step12/final_selection.json':
        'e495222d6050492784b334110973b219dce2d3e9deaf516d311878462c6e47b6',
    'models/final_logistic_regression.joblib':
        '498112adf28d66f22f84f76101187c7a94eefeda62b7a92b45f1f9152790e097',
}


def in_colab():
    return 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))


def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def ensure_analysis_packages():
    # Do not replace Colab's IPython, ipykernel, or notebook server.
    changed = {name for name, (_, version) in ANALYSIS_PACKAGES.items()
               if installed_version(name) != version}
    loaded_before = {name for name, (module, _) in ANALYSIS_PACKAGES.items()
                     if module in sys.modules}
    if changed:
        print('Installing pinned analysis packages. This may take a few minutes.', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                        *[f'{name}=={version}' for name, (_, version)
                          in ANALYSIS_PACKAGES.items()]], check=True)
    for name, (_, version) in ANALYSIS_PACKAGES.items():
        if installed_version(name) != version:
            raise RuntimeError(f'{name} installation did not reach {version}; stop and inspect pip output.')
    stale = changed & loaded_before
    for name, (module, version) in ANALYSIS_PACKAGES.items():
        loaded = sys.modules.get(module)
        if loaded is not None and getattr(loaded, '__version__', version) != version:
            stale.add(name)
    if stale:
        raise RuntimeError(
            'SETUP PAUSED: packages already loaded in memory need a restart: '
            + ', '.join(sorted(stale))
            + '. Choose Runtime > Restart session, then Runtime > Run all. '
              'Do not disconnect/delete the runtime. Installed packages are retained.')


def repository_at_or_above(folder):
    folder = Path(folder).resolve()
    for candidate in (folder, *folder.parents):
        if ((candidate / 'src/eligibility.py').is_file()
                and (candidate / 'data/splits/step6_split_plan.json').is_file()
                and (candidate / 'notebooks').is_dir()):
            return candidate
    return None


def checked_checkout(destination):
    destination = Path(destination)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--no-checkout', SOURCE_URL, str(destination)], check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', SOURCE_COMMIT], check=True)
    elif not (destination / '.git').is_dir():
        raise RuntimeError('The setup folder already exists but is not a Git checkout. '
                           'Use a fresh Colab runtime; no existing files were overwritten.')
    head = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    origin = subprocess.check_output(['git', '-C', str(destination), 'remote', 'get-url', 'origin'], text=True).strip()
    if head != SOURCE_COMMIT or origin != SOURCE_URL:
        raise RuntimeError('Existing checkout has a different source/version. '
                           'Use a fresh runtime. Setup will not reset or overwrite it.')
    return destination


def verify_inputs(root):
    for relative, expected in PROTECTED_FILES.items():
        path = Path(root) / relative
        if not path.is_file():
            raise FileNotFoundError(f'Missing required project input: {relative}')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Protected input differs: {relative}. No replacement was attempted.')


def prepare_colab():
    if not in_colab():
        # The ordinary local/verification workflow already supplies its environment.
        return None
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise RuntimeError('These package pins need Python 3.12 or 3.13. '
                           'Use a compatible Colab runtime, or the documented local Python 3.12 setup.')
    ensure_analysis_packages()
    root = repository_at_or_above(Path.cwd())
    if root is None:
        root = checked_checkout(Path('/content/cse437-colab') / SOURCE_COMMIT)
        print('Analysis source commit:', SOURCE_COMMIT)
    else:
        print('Using the existing project working directory; no checkout reset or update.')
    verify_inputs(root)
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Setup ready:', root)
    print('CPU runtime is sufficient. No Google Drive mount or manual data upload is needed.')
    print('Development scores may vary; notebook 05 verifies the original cached test results.')
    return root


if __name__ == '__main__':
    prepare_colab()


In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
OUT=ROOT/"data/results/step13"
summary=json.loads((OUT/"evaluation_summary.json").read_text())
metrics=pd.read_csv(OUT/"final_metrics.csv").iloc[0]
groups=pd.read_csv(OUT/"subgroup_metrics.csv")
probability=pd.read_csv(OUT/"probability_diagnostics.csv")
errors=pd.read_csv(OUT/"error_examples.csv")
coefficients=pd.read_csv(OUT/"feature_coefficients.csv")
print(json.dumps({"selection":summary["selection"],"official_test_evaluations":summary["official_test_evaluations"],"development_rows_fitted":summary["development_rows_fitted"],"test_rows_evaluated":summary["test_rows_evaluated"]},indent=2))


{
  "selection": {
    "family": "logistic_regression",
    "representation": "selected",
    "search_parameters": {
      "model__C": 1.0,
      "model__class_weight": "balanced"
    },
    "threshold": 0.5,
    "mean_development_f1": 0.7321017246390454
  },
  "official_test_evaluations": 1,
  "development_rows_fitted": 95415,
  "test_rows_evaluated": 23795
}


## Frozen final-test result

The test window is 2017-04-23 through 2017-08-31. It was excluded from feature/model/tuning decisions. Mean development F1 (0.732102) is shown only as context; the single held-out result is not a confidence interval or proof of future performance.

![Final test performance and confusion matrix](../figures/09_final_test_performance.png)


In [2]:
print(metrics[["f1","accuracy","precision","recall","roc_auc","brier_score","test_rows","test_cancellations","test_cancellation_rate","tn","fp","fn","tp"]].to_string())
assert int(metrics.tn+metrics.fp+metrics.fn+metrics.tp)==int(metrics.test_rows)==23795
assert int(metrics.fn+metrics.tp)==int(metrics.test_cancellations)
print("\nDevelopment mean F1:",round(summary["selection"]["mean_development_f1"],6))
print("Held-out F1:",round(metrics.f1,6))
print("Model/threshold reselected after test:",summary["model_reselected_from_test"],summary["threshold_changed_after_test"])


f1                            0.750592
accuracy                      0.760874
precision                     0.654187
recall                        0.880321
roc_auc                       0.875977
brier_score                   0.160007
test_rows                 23795.000000
test_cancellations         9726.000000
test_cancellation_rate        0.408741
tn                         9543.000000
fp                         4526.000000
fn                         1164.000000
tp                         8562.000000

Development mean F1: 0.732102
Held-out F1: 0.750592
Model/threshold reselected after test: False False


## Where the model fails

Balanced weighting favors detecting cancellations: recall is high, but false positives exceed false negatives. Subgroup scores below are descriptive after test access and cannot justify changing the frozen pipeline. Counts depend on group size; small groups must not be overinterpreted.

The fixed-bin probability curve lies below the diagonal: predicted probabilities exceed observed rates in every populated bin. This is consistent with balanced class weighting and means these outputs should not be presented as calibrated probabilities. No post-test calibration or threshold adjustment is performed.

![Probability and hotel error diagnostics](../figures/10_final_error_analysis.png)


In [3]:
for dimension in ["hotel","lead_time_band","deposit_type","market_segment","customer_type"]:
    view=groups.loc[groups.dimension.eq(dimension),["group","rows","cancellations","error_rate","f1","precision","recall","fp","fn"]]
    print("\n"+dimension+":")
    print(view.round(6).to_string(index=False))
assert groups.groupby("dimension").rows.sum().eq(23795).all()
print("\nEvery subgroup dimension reconciles to all 23,795 test bookings.")



hotel:
       group  rows  cancellations  error_rate       f1  precision   recall   fp  fn
Resort Hotel  7459           2666    0.202440 0.741084   0.682565 0.810578 1005 505
  City Hotel 16336           7060    0.255877 0.753857   0.645132 0.906657 3521 659

lead_time_band:
  group  rows  cancellations  error_rate       f1  precision   recall   fp  fn
  31–90  3995           1571    0.275344 0.709456   0.606321 0.854870  872 228
181–365  5769           2897    0.242676 0.792777   0.693962 0.924405 1181 219
 91–180  7564           3617    0.251719 0.781401   0.668172 0.940835 1690 214
    0–7  2388            259    0.175042 0.334395   0.284553 0.405405  264 154
   8–30  3011            933    0.272003 0.588235   0.553977 0.627010  471 348
   366+  1068            449    0.045880 0.948148   0.903226 0.997773   48   1

deposit_type:
     group  rows  cancellations  error_rate       f1  precision   recall   fp   fn
No Deposit 21482           7415    0.264454 0.687840   0.580397 0.844100

## Concrete wrong predictions

The protocol selects, separately for false positives and false negatives, five most-confident errors and five additional errors closest to 0.5. These are deliberately diagnostic slices, not random or representative samples. `error_examples.csv` includes all 20 rows; no reservation-status leakage fields are exported.


In [4]:
print(errors.to_string(index=False))
assert len(errors)==20 and not errors.source_row_id.duplicated().any()
print("\nError-type counts:")
print(errors.groupby(["error_type","example_reason"]).size().to_string())


         example_reason  source_row_id arrival_date        hotel  lead_time lead_time_band deposit_type market_segment   customer_type  previous_cancellations  total_of_special_requests  actual  predicted  cancellation_probability error_type
      most_confident_FP          14182   2017-06-09 Resort Hotel         80          31–90   No Deposit         Direct       Transient                       1                          0       0          1                  0.999603         FP
      most_confident_FP         117788   2017-08-04   City Hotel        211        181–365   No Deposit      Online TA       Transient                       0                          0       0          1                  0.992440         FP
      most_confident_FP         119068   2017-08-28   City Hotel        311        181–365   No Deposit      Online TA       Transient                       0                          0       0          1                  0.990008         FP
      most_confident_FP         

## Model inspection and limits

The strongest positive coefficient is the Non Refund indicator; many large coefficients are particular agent categories. Required parking spaces has a large negative coefficient. These are associations in an encoded linear model—not causal effects or universally comparable feature importance. Sparse categories, source timing, repeated profiles and temporal change can make coefficients unstable.

The full-development fit retains 406 encoded columns. Because the selection rule is refitted on all development rows, that width need not match individual CV-fold widths. `models/final_logistic_regression.joblib` contains the entire fitted preprocessing, selection and classifier pipeline.


In [5]:
model_path=ROOT/"models/final_logistic_regression.joblib"
model=joblib.load(model_path)
assert model.named_steps["model"].get_params(deep=False)["C"]==1.0
assert model.named_steps["model"].get_params(deep=False)["class_weight"]=="balanced"
assert len(coefficients)==summary["encoded_features"]==406
assert hashlib.sha256(model_path.read_bytes()).hexdigest()==summary["output_sha256"]["models/final_logistic_regression.joblib"]
print(coefficients.head(20).to_string(index=False))
print("\nSaved pipeline verified:",model_path.relative_to(ROOT))


                             feature  coefficient  absolute_coefficient                 direction
categorical__deposit_type_Non Refund     2.929658              2.929658 higher cancellation score
               categorical__agent_89    -2.783724              2.783724  lower cancellation score
              categorical__agent_152    -2.744587              2.744587  lower cancellation score
               categorical__agent_17     2.582659              2.582659 higher cancellation score
               categorical__agent_11    -2.566355              2.566355  lower cancellation score
numeric__required_car_parking_spaces    -2.290799              2.290799  lower cancellation score
               categorical__agent_69    -2.138171              2.138171  lower cancellation score
              categorical__agent_201    -2.114686              2.114686  lower cancellation score
              categorical__agent_308    -2.061380              2.061380  lower cancellation score
              catego

## Evidence integrity

Descriptive associations, model coefficients, development comparisons and held-out performance answer different questions. The frozen results must not be used to retune, recalibrate, change the threshold or switch models.

In [6]:
for relative,expected in summary["output_sha256"].items():
    actual=hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()
    assert actual==expected,relative
assert summary["official_test_evaluations"]==1
assert summary["representation_unchanged_after_test_prediction"]
assert not summary["model_reselected_from_test"] and not summary["threshold_changed_after_test"]
print("Step 13 evidence hashes verified. Official test result is frozen.")


Step 13 evidence hashes verified. Official test result is frozen.


## Supplementary test comparison

The baseline/Random Forest comparison was added after the Logistic Regression test result was known. It is reporting-only, not a preregistered simultaneous test. This cell requires the original cached evidence and verifies the original selection hash. It cannot retrain a missing comparison.

**Reproducibility status:** an earlier full run failed because notebook 04 overwrote the frozen selection with a different development winner. The repaired notebook 04 writes to a separate workspace and reports differences explicitly. Existing outputs remain the published reference; fresh-kernel validation of the repair is pending. The numerical cause of the earlier score differences has not been established.

In [7]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.test_comparison import run_test_comparison
comparison, supplement = run_test_comparison(ROOT, require_cached=True)
assert supplement["selected_model"] == "logistic_regression"
assert not supplement["model_reselected_from_test"]
assert not supplement["threshold_changed"]
assert supplement["frozen_step13_artifacts_unchanged"]
print(comparison.to_string(index=False))
print("Step 15 reporting supplement verified; selected model unchanged.")
print("Added after Step 13 results; not a simultaneous preregistered comparison.")


              model  selected_before_test  test_rows  threshold       f1  accuracy  precision   recall  roc_auc    tn   fp   fn   tp  brier_score
  majority_baseline                 False      23795        0.5 0.000000  0.591259   0.000000 0.000000 0.500000 14069    0 9726    0     0.408741
logistic_regression                  True      23795        0.5 0.750592  0.760874   0.654187 0.880321 0.875977  9543 4526 1164 8562     0.160007
      random_forest                 False      23795        0.5 0.723115  0.789325   0.781239 0.673041 0.878190 12236 1833 3180 6546     0.145476
Step 15 reporting supplement verified; selected model unchanged.
Added after Step 13 results; not a simultaneous preregistered comparison.
